# 01 EDA und Feature-Auswahl

Ziel: Dry-Bean-Datensatz laden, Klassenverteilung und Feature-Korrelationen untersuchen und eine erste Auswahl von 5 bis 10 Features begruenden.

## Forschungsfrage

Inwiefern lassen sich Dry-Bean-Sorten anhand weniger morphologischer Bildmerkmale zuverlaessig klassifizieren, und welche Merkmale erklaeren die Entscheidungen verschiedener Multiclass-Klassifikatoren am staerksten?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## Daten laden

Der Datensatz wird direkt aus dem UCI Machine Learning Repository geladen.

In [ ]:
dry_bean = fetch_ucirepo(id=602)
X = dry_bean.data.features
y = dry_bean.data.targets.squeeze()
df = X.copy()
df["Class"] = y

df.head()

In [ ]:
df.shape, df.isna().sum().sum(), df["Class"].value_counts()

## Klassenverteilung

Die Klassenverteilung ist wichtig, weil Accuracy bei unausgeglichenen Klassen alleine nicht ausreicht. Deshalb wird spaeter auch Macro-F1 verwendet.

In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(data=df, x="Class", order=df["Class"].value_counts().index)
plt.title("Klassenverteilung im Dry Bean Dataset")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Korrelationen

Stark korrelierte Features liefern oft aehnliche Information. Fuer eine interpretierbare Feature-Auswahl sollten redundante Groessenmerkmale nicht alle gleichzeitig verwendet werden.

In [ ]:
corr = X.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
plt.title("Korrelationsmatrix der Features")
plt.tight_layout()
plt.show()

In [ ]:
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_corr"})
    .query("abs_corr >= 0.90")
    .sort_values("abs_corr", ascending=False)
)
high_corr_pairs

## Erste Feature-Auswahl

Vorschlag fuer eine erste, interpretierbare Auswahl:

- `Area`: Groesse der Bohne
- `AspectRation`: Verhaeltnis von Laenge zu Breite
- `Eccentricity`: Rundheit bzw. Laenglichkeit
- `Compactness`: Kompaktheit der Form
- `Roundness`: Rundheitsmerkmal
- `ShapeFactor1`, `ShapeFactor2`, `ShapeFactor3`: ergaenzende Formfaktoren

Diese Auswahl kombiniert Groesse und Form, vermeidet aber mehrere stark redundante Groessenfeatures wie `Area`, `ConvexArea` und `EquivDiameter` gleichzeitig.

In [ ]:
selected_features = [
    "Area",
    "AspectRation",
    "Eccentricity",
    "Compactness",
    "Roundness",
    "ShapeFactor1",
    "ShapeFactor2",
    "ShapeFactor3",
]

df[selected_features + ["Class"]].head()